# Tree-based Model comparisons - RandomForest+SMOTE vs BalancedRandomForest vs XGBoost vs LightGBM

In [ ]:
!pip install lightgbm
# !pip install --upgrade lightgbm
!pip install "numpy<2.0"

# Notes for which numpy version to reinstall later
# Found existing installation: numpy 2.0.2
#     Uninstalling numpy-2.0.2:
#       Successfully uninstalled numpy-2.0.2

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import seaborn as sns
import os
import duckdb

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


In [2]:
cleaned_parquet = "../data/cleaned_data/cleaned_fraud.parquet"

print(f"📊 Original size: {os.path.getsize(cleaned_parquet) / (1024**3):.2f} GB")

con = duckdb.connect()

📊 Original size: 0.14 GB


In [3]:
# Load full dataset
df = con.execute(f"SELECT * FROM '{cleaned_parquet}'").fetch_df()

# 1. EDA & Pattern Detection
print("Original Class Distribution:")
print(df['is_fraud'].value_counts(normalize=True))

# Check for negative time values (Pattern Detection)
neg_time = df[df['time_since_last_transaction'] < 0]
print(f"\nRows with negative time_since_last_transaction: {len(neg_time)}")
if len(neg_time) > 0:
    print("Negative values might indicate data errors or specific flags. Treating as valid numeric for now.")
    

con.close()

Original Class Distribution:
is_fraud
False    0.956244
True     0.043756
Name: proportion, dtype: float64

Rows with negative time_since_last_transaction: 2051331
Negative values might indicate data errors or specific flags. Treating as valid numeric for now.


In [4]:
# We reduce sample size for faster experimentation - reduced to 2% of the entire dataset
from sklearn.model_selection import train_test_split

# Desired sample fraction
sample_frac = 0.2  # 2%

# Perform stratified sampling
df_sample, _ = train_test_split(
    df,
    train_size=sample_frac,
    stratify=df['is_fraud'],
    random_state=42
)

print(df_sample.shape)
print(df_sample['is_fraud'].value_counts(normalize=True))

(820697, 20)
is_fraud
False    0.956243
True     0.043757
Name: proportion, dtype: float64


In [5]:


# 2. Data Preparation for SMOTE
# - Drop high cardinality identifiers
# - Encode categorical variables
# - Scale numerical variables

categorical_cols = ['transaction_type', 'merchant_category', 'location', 'device_used', 'payment_channel']
numerical_cols = ['amount', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'hour', 'day_of_week']
drop_cols = ['sender_account', 'receiver_account', 'ip_address', 'device_hash', 'year', 'month', 'day_of_month'] # high-cardinality identifiers that do not generalize well and can negatively affect the model training. Dropping date because we already extracted more meaningful features like hours and days of the week

# Separate Features and Target
X = df_sample.drop(columns=['is_fraud'] + drop_cols, errors='ignore')
y = df_sample['is_fraud']

# Split Data (Best Practice: Split BEFORE SMOTE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # stratifying by y to ensure both splits have an equal amount of the target variable y

# Preprocessing Pipeline 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]
)



In [6]:
# 4. Full Pipeline: Preprocess → SMOTE → Model - automatic preprocessing applying SMOTE in CV
# Random Forest Classifier with SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

rf_smote_pipeline = ImbPipeline([
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        n_jobs=-1,
        random_state=42
    ))

])


In [7]:
# BalancedRandomForest (no SMOTE)
from imblearn.ensemble import BalancedRandomForestClassifier

brf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', BalancedRandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        n_jobs=-1,
        random_state=42
    ))
])

In [8]:
# XGBoost Classifier with scale_pos_weight
from xgboost import XGBClassifier

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

xgb_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=42
    ))
])

In [9]:
# LightGBM Classifier with is_unbalance

from lightgbm import LGBMClassifier

lgbm_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        random_state=42,
        n_jobs=-1
    ))
])

In [10]:
# Fit all models and build comparison table
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, precision_recall_curve, auc
)
import pandas as pd

models = {
    "RandomForest+SMOTE": rf_smote_pipeline,
    "BalancedRandomForest": brf_pipeline,
    "XGBoost": xgb_pipeline,
    "LightGBM": lgbm_pipeline,
}

results = []

for name, pipe in models.items():
    print(f"Training {name}...")
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(rec_curve, prec_curve)
    
    results.append({
        "model": name,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    })

results_df = pd.DataFrame(results).set_index("model")
results_df

Training RandomForest+SMOTE...


c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Training BalancedRandomForest...


c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `False` to silence this warni

Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Number of positive: 28729, number of negative: 627828
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017781 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 977
[LightGBM] [Info] Number of data points in the train set: 656557, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.043757 -> initscore=-3.084359
[LightGBM] [Info] Start training from score -3.084359


c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\J2F\anaconda3\envs\dsi_participant\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,f1,recall,precision,roc_auc,pr_auc
model,,,,,
RandomForest+SMOTE,0.078928,0.434419,0.043407,0.498093,0.045955
BalancedRandomForest,0.081817,0.545530,0.044225,0.499507,0.043481
XGBoost,0.081402,0.511000,0.044223,0.500904,0.043895
LightGBM,0.079722,0.445280,0.043780,0.501678,0.044199
